In [1]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 34.74,
    "longitude": 10.76,
    "current": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
        "weather_code",
        "wind_speed_10m"
    ]
}

response = requests.get(url, params=params)

print(response.status_code)

data = response.json()

print(data)

200
{'latitude': 34.75, 'longitude': 10.75, 'generationtime_ms': 0.06878376007080078, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 11.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm', 'rain': 'mm', 'weather_code': 'wmo code', 'wind_speed_10m': 'km/h'}, 'current': {'time': '2026-09-15T20:45', 'interval': 900, 'temperature_2m': 25.4, 'relative_humidity_2m': 76, 'precipitation': 0.0, 'rain': 0.0, 'weather_code': 1, 'wind_speed_10m': 2.9}}


In [4]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 34.74,
    "longitude": 10.76,
    "current": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
        "weather_code",
        "wind_speed_10m"
    ]
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

weather = data["current"]

print("Temperature:", weather["temperature_2m"], "°C")
print("Humidity:", weather["relative_humidity_2m"], "%")
print("Wind:", weather["wind_speed_10m"], "km/h")
print("Rain:", weather["rain"], "mm")
print("Weather code:", weather["weather_code"])

Temperature: 25.3 °C
Humidity: 77 %
Wind: 2.9 km/h
Rain: 0.0 mm
Weather code: 1


In [1]:
import requests

url = "https://geocoding-api.open-meteo.com/v1/search"

params = {
    "name": "Sfax",
    "count": 5,
    "language": "en",
    "format": "json"
}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()

data = response.json()

for place in data.get("results", []):
    print(
        place["name"],
        place.get("country"),
        place["latitude"],
        place["longitude"]
    )

Sfax Tunisia 34.74056 10.76028
Sfax–Thyna International Airport Tunisia 34.71795 10.69097


In [2]:
print(data)

{'results': [{'id': 2467454, 'name': 'Sfax', 'latitude': 34.74056, 'longitude': 10.76028, 'elevation': 8.0, 'feature_code': 'PPLA', 'country_code': 'TN', 'admin1_id': 2467450, 'admin2_id': 2467453, 'admin3_id': 11904494, 'timezone': 'Africa/Tunis', 'population': 280566, 'country_id': 2464461, 'country': 'Tunisia', 'admin1': 'Sfax Governorate', 'admin2': 'Sfax Ville', 'admin3': 'La Medina'}, {'id': 6296475, 'name': 'Sfax–Thyna International Airport', 'latitude': 34.71795, 'longitude': 10.69097, 'elevation': 25.0, 'feature_code': 'AIRP', 'country_code': 'TN', 'admin1_id': 2467450, 'admin2_id': 11104959, 'admin3_id': 11905459, 'timezone': 'Africa/Tunis', 'country_id': 2464461, 'country': 'Tunisia', 'admin1': 'Sfax Governorate', 'admin2': 'Tina', 'admin3': 'El Hajeb'}], 'generationtime_ms': 0.14770031}


In [3]:
import requests


def get_weather(city):
    # 1. Convert city name -> coordinates
    geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    geo_response = requests.get(
        geocoding_url,
        params=geo_params,
        timeout=10
    )

    geo_response.raise_for_status()

    geo_data = geo_response.json()

    if not geo_data.get("results"):
        return {"error": f"Location '{city}' not found"}

    location = geo_data["results"][0]

    latitude = location["latitude"]
    longitude = location["longitude"]

    # 2. Get weather using coordinates
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "rain",
            "weather_code",
            "wind_speed_10m"
        ]
    }

    weather_response = requests.get(
        weather_url,
        params=weather_params,
        timeout=10
    )

    weather_response.raise_for_status()

    weather_data = weather_response.json()
    current = weather_data["current"]

    # 3. Return clean data
    return {
        "city": location["name"],
        "country": location.get("country"),
        "latitude": latitude,
        "longitude": longitude,
        "temperature": current["temperature_2m"],
        "humidity": current["relative_humidity_2m"],
        "precipitation": current["precipitation"],
        "rain": current["rain"],
        "wind_speed": current["wind_speed_10m"],
        "weather_code": current["weather_code"]
    }


print(get_weather("Sfax"))

{'city': 'Sfax', 'country': 'Tunisia', 'latitude': 34.74056, 'longitude': 10.76028, 'temperature': 25.2, 'humidity': 78, 'precipitation': 0.0, 'rain': 0.0, 'wind_speed': 3.0, 'weather_code': 1}


In [2]:
import requests


def get_weather(city: str) -> dict:
    """
    Get the current weather for a city using Open-Meteo.

    The function:
    1. Converts the city name to latitude/longitude.
    2. Retrieves current weather using those coordinates.
    3. Returns clean structured data.
    """

    # -------------------------
    # 1. Geocoding
    # -------------------------

    geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json",
    }

    geo_response = requests.get(
        geocoding_url,
        params=geo_params,
        timeout=10,
    )

    geo_response.raise_for_status()

    geo_data = geo_response.json()

    if not geo_data.get("results"):
        return {
            "error": f"Location '{city}' not found."
        }

    location = geo_data["results"][0]

    latitude = location["latitude"]
    longitude = location["longitude"]

    # -------------------------
    # 2. Weather
    # -------------------------

    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "rain",
            "weather_code",
            "wind_speed_10m",
        ],
    }

    weather_response = requests.get(
        weather_url,
        params=weather_params,
        timeout=10,
    )

    weather_response.raise_for_status()

    weather_data = weather_response.json()

    current = weather_data["current"]

    # -------------------------
    # 3. Clean result
    # -------------------------

    return {
        "city": location["name"],
        "country": location.get("country"),
        "latitude": latitude,
        "longitude": longitude,
        "temperature_c": current["temperature_2m"],
        "humidity_percent": current["relative_humidity_2m"],
        "precipitation_mm": current["precipitation"],
        "rain_mm": current["rain"],
        "wind_speed_kmh": current["wind_speed_10m"],
        "weather_code": current["weather_code"],
    }

print(get_weather("Sfax"))

{'city': 'Sfax', 'country': 'Tunisia', 'latitude': 34.74056, 'longitude': 10.76028, 'temperature_c': 25.2, 'humidity_percent': 78, 'precipitation_mm': 0.0, 'rain_mm': 0.0, 'wind_speed_kmh': 3.0, 'weather_code': 1}


In [4]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

In [5]:
from tools.weather import get_weather

weather = get_weather("Sfax")

print(weather)

ImportError: cannot import name 'get_weather' from 'tools.weather' (c:\Users\Yessine\Desktop\vehicle_ai_copilote\tools\weather.py)